In [5]:
import pandas as pd
import numpy as np

# preprocessing
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from imblearn.under_sampling import RandomUnderSampler

# model machine learning
from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,log_loss
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [6]:
df = pd.read_csv('../0.dataset/dataset_Heart_attack_clean.csv')
df_x = df.drop(columns='heart_attack')
df_y = df[['heart_attack']]

# 1.Binary Classification K-Nearest Neighbors

In [7]:
feature_numerik = ['age','body_mass_index','systolic_blood_pressure','forced_expiratory_volume_1','time_to_event_or_censoring']
feature_categori = ['gender']

preprocessor = ColumnTransformer(
    transformers=[
        ('numerik_scaler',StandardScaler(),feature_numerik),
        ('categori_encoding',OneHotEncoder(),feature_categori)
        ],
   remainder='passthrough' # Kolom kategori yang sudah ter-encode dari sananya akan aman lewat lewat sini
)

In [8]:
selector_model = KNeighborsClassifier()
model_KNeighborsClassifier = Pipeline([
    ('preprocessing',preprocessor),
    ('undersampling',RandomUnderSampler(random_state=42)),
    ('feature_selection',SequentialFeatureSelector(estimator=selector_model,direction='forward',cv=3)),
    ('model_classifier',KNeighborsClassifier())
])

In [9]:
params = {
 'feature_selection__n_features_to_select': [13, 14,'auto',5,9,7],
 'model_classifier__n_neighbors' : list(range(5,21,2)),
 'model_classifier__weights': ['uniform','distance'],
 'model_classifier__metric': ['euclidean','manhattan', 'minkowski'],
 'model_classifier__p': [1,2,3]
}
X_train,X_test,y_train,y_test = train_test_split(df_x,df_y,test_size=0.2,random_state=42,stratify=df_y)

random_search = RandomizedSearchCV(estimator=model_KNeighborsClassifier,param_distributions=params,n_iter=10,cv=5,scoring='accuracy',random_state=42,n_jobs=-1,verbose=3)
random_search.fit(X_train,y_train.values.ravel())

Fitting 5 folds for each of 10 candidates, totalling 50 fits


,estimator,Pipeline(step...lassifier())])
,param_distributions,"{'feature_selection__n_features_to_select': [13, 14, ...], 'model_classifier__metric': ['euclidean', 'manhattan', ...], 'model_classifier__n_neighbors': [5, 7, ...], 'model_classifier__p': [1, 2, ...], ...}"
,n_iter,10
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [10]:
best_model_pipeline = random_search.best_estimator_
preprocessor_step = best_model_pipeline.named_steps['preprocessing']
sfs_step = best_model_pipeline.named_steps['feature_selection']

kolom_setelah_transformasi = preprocessor_step.get_feature_names_out()
fitur_terpilih = kolom_setelah_transformasi[sfs_step.get_support()]

print(f'\nHyperparameter Terbaik:\n{random_search.best_params_}')
print(f'\nFitur Terbaik Yang Terpilih:\n{list(fitur_terpilih)}')

#cek score
tes_accuracy = best_model_pipeline.score(X_test, y_test)

y_pred_test = best_model_pipeline.predict(X_test)
y_prob_test = best_model_pipeline.predict_proba(X_test)

y_pred_train = best_model_pipeline.predict(X_train)
y_prob_train = best_model_pipeline.predict_proba(X_train)

print(f'\nAkurasi Validasi Terbaik: {random_search.best_score_*100:.2f}%')
print(f'\nAkurasi Data Test: {tes_accuracy*100:.2f}%')


Hyperparameter Terbaik:
{'model_classifier__weights': 'uniform', 'model_classifier__p': 1, 'model_classifier__n_neighbors': 7, 'model_classifier__metric': 'manhattan', 'feature_selection__n_features_to_select': 14}

Fitur Terbaik Yang Terpilih:
['numerik_scaler__age', 'numerik_scaler__body_mass_index', 'numerik_scaler__forced_expiratory_volume_1', 'numerik_scaler__time_to_event_or_censoring', 'categori_encoding__gender_F', 'categori_encoding__gender_M', 'remainder__smoker', 'remainder__hypertension_treated', 'remainder__family_history_of_cardiovascular_disease', 'remainder__atrial_fibrillation', 'remainder__chronic_kidney_disease', 'remainder__rheumatoid_arthritis', 'remainder__diabetes', 'remainder__chronic_obstructive_pulmonary_disorder']

Akurasi Validasi Terbaik: 66.08%

Akurasi Data Test: 69.06%


In [11]:
def evaluate_model(pred,test,prob,evaluate,model_name='K-Nearest Neighbors'):
    accuracy = accuracy_score(test,pred)
    precision = precision_score(test,pred)
    recall = recall_score(test,pred)
    f1 = f1_score(test,pred)
    roc_auc = roc_auc_score(test,prob[:,1])
    logloss = log_loss(test,prob)

    data = {
        'Model': [model_name],
        'Evaluated': [evaluate],
        'Accuracy': [accuracy],
        'Precision': [precision],
        'Recall': [recall],
        'F1-Score': [f1],
        'ROC-AUC': [roc_auc],
        'Log Loss': [logloss]
    }

    df_result = pd.DataFrame(data)
    return df_result

In [ ]:
def check_model_fit(df_eval_train,df_eval_test):
    df_combined = pd.concat([df_eval_train,df_eval_test],ignore_index=True)
    accuracy_train = df_eval_train['Accuracy'].values[0]
    accuracy_test = df_eval_test['Accuracy'].values[0]
    gap = accuracy_train - accuracy_test

    if accuracy_train < 0.60 or accuracy_test <0.60:
        status = 'Undeerfitting (Akurasi Rendah)'
    elif gap > 0.05:
        status = f'Overfitting (gap:{gap*100:.2f})'
    elif gap < -0.05:
        status = 'Overfitting / Data Leakage (Test > Train)'
    else:
        status = 'Good Fit'

    df_combined['Status'] = status
    return df_combined

In [13]:
df_eval_train = evaluate_model(y_pred_train,y_train,y_prob_train,evaluate='Training')
df_eval_test = evaluate_model(y_pred_test,y_test,y_prob_test,evaluate='Test')
df_eval = check_model_fit(df_eval_train,df_eval_test)

print('================================= PREDIKSI DENGAN SAMPLE DATASET ===================================')
print(df_eval.to_string())
print("\n" + "="*100 + "\n")

================================= PREDIKSI DENGAN SAMPLE DATASET ===================================
                 Model Evaluated  Accuracy  Precision    Recall  F1-Score   ROC-AUC  Log Loss    Status
0  K-Nearest Neighbors  Training  0.694072   0.016110  0.794444  0.031580  0.806680  1.408849  Good Fit
1  K-Nearest Neighbors      Test  0.690617   0.015391  0.766667  0.030177  0.791475  1.425621  Good Fit




## Melakukan Prediksi Dengan Data Random

In [14]:
data_4_pasien = {
'gender': ['F', 'F', 'F', 'F'],
    'age': [31, 72, 28, 63],
    'body_mass_index': [25.9, 28.6, 27.0, 27.0],
    'smoker': [0, 0, 0, 0],  # Diasumsikan dari data numerik biner Anda
    'systolic_blood_pressure': [112.0, 125.0, 102.0, 130.0],
    'hypertension_treated': [1, 1, 0, 0],
    'family_history_of_cardiovascular_disease': [0, 0, 0, 0],
    'atrial_fibrillation': [0, 0, 0, 0],
    'chronic_kidney_disease': [0, 1, 0, 0],
    'rheumatoid_arthritis': [0, 0, 0, 0],
    'diabetes': [0, 0, 0, 0],
    'chronic_obstructive_pulmonary_disorder': [0, 0, 0, 0],
    'forced_expiratory_volume_1': [90.96, 88.00, 98.37, 100.00], # Mengisi nilai representatif berdasarkan teks data Anda
    'time_to_event_or_censoring': [9833, 100, 6954, 100],        # Mengisi nilai representatif berdasarkan teks data Anda
    'heart_attack': [1, 0, 1, 0]
}
data_pasien_baru_x = pd.DataFrame(data_4_pasien)
data_pasien_baru_y = data_pasien_baru_x['heart_attack']

In [15]:
print("=== Melakukan Prediksi Data Pasien Baru ===")
prediksi_pasien = best_model_pipeline.predict(data_pasien_baru_x)
probabilitas_pasien = best_model_pipeline.predict_proba(data_pasien_baru_x)

hasil_df = data_pasien_baru_x.copy()
hasil_df['Prediksi Serangan Jantung'] = prediksi_pasien
hasil_df['Peluang Aman(%)'] = probabilitas_pasien[:,0] * 100
hasil_df['Peluang Terkena (%)'] = probabilitas_pasien[:,1] * 100

akurasi_bawaan = best_model_pipeline.score(data_pasien_baru_x, data_pasien_baru_y)
df_eval_data_baru = evaluate_model(
    pred=prediksi_pasien, 
    test=data_pasien_baru_y, 
    prob=probabilitas_pasien, 
    evaluate='Data_Baru'
)

print(f"Akurasi Model: {akurasi_bawaan * 100:.2f}%")
print("\nTabel Skor Evaluasi Lengkap Data Baru:")
print(df_eval_data_baru.to_string(index=False))
hasil_df[['Prediksi Serangan Jantung', 'Peluang Aman(%)', 'Peluang Terkena (%)']]

=== Melakukan Prediksi Data Pasien Baru ===
Akurasi Model: 0.00%

Tabel Skor Evaluasi Lengkap Data Baru:
              Model Evaluated  Accuracy  Precision  Recall  F1-Score  ROC-AUC  Log Loss
K-Nearest Neighbors Data_Baru       0.0        0.0     0.0       0.0      0.0   1.49797


,Prediksi Serangan Jantung,Peluang Aman(%),Peluang Terkena (%)
0,0,85.714286,14.285714
1,1,14.285714,85.714286
2,0,71.428571,28.571429
3,1,42.857143,57.142857


# 2.Multiclass Classification K-Nearest Neighbors